## Transformation Guide to SysML v2 Workflows

### 1. Introduction

This notebook presents the transformation pipeline used to convert existing workflow representations into SysML v2 models. The objective is to provide a deterministic and reproducible approach for generating structured SysML v2 workflows.

The transformations are applied to multiple source datasets, including graph-based workflows **WorfBench**, BPMN models **SAP-SAM**, and SOP-based descriptions **SOP-Bench**. Each workflow is parsed to extract its core components (actions, dependencies, and control structures), which are then mapped into equivalent SysML v2 constructs such as actions, decisions, forks, and joins.

This notebook serves as a practical guide to illustrate the transformation logic, and provide examples of the resulting SysML v2 representations.

The implementation avoids the use of LLMs to ensure reproducibility and prevent semantic inconsistencies during the transformation process.

### Helper : Loading Python Modules from File Paths

In [ ]:
from pathlib import Path
import importlib.util
import json
import sys

ROOT = Path.cwd()

def load_module(module_name: str, file_path: Path):
    spec = importlib.util.spec_from_file_location(module_name, str(file_path))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

### 2. Transformation of WorfBench:

**A. Load the dataset:**

In [ ]:
from datasets import load_dataset

ds = load_dataset("zjunlp/WorFBench_train")

print("Splits disponibles:", ds.keys())
print("Nombre d'instances train:", len(ds["train"]))
print("Champs:", ds["train"].column_names)

**B. Extract user prompt, source, nodes and edges:**

In [ ]:
worf_dir = ROOT / "WorfBench"
if str(worf_dir) not in sys.path:
    sys.path.insert(0, str(worf_dir))

worfbench = load_module("worfbench_module", worf_dir / "run_transformation.py")

row_index = 0
instance = ds["train"][row_index]

user_prompt, source = worfbench.extract_user_prompt_and_source(instance)
graph = worfbench.extract_nodes_and_edges(instance)

print("source:", source)
print("user_prompt:", user_prompt)
print("nodes:")
for node_id, node_text in graph["nodes"].items():
    print(node_id, "->", node_text)

print("\nedges:")
for src, dst in graph["edges"]:
    print(src, "->", dst)

**C. Transform into SysML:**

In [ ]:
from Transformers.graph_to_sysml import transform_into_sysml

sysml_code = transform_into_sysml(
    graph,
    package_name="WorFBenchPackage",
    process_name="WorFBenchProcess"
)

print(sysml_code)

### 3. Transformation of SAP-SAM:

**A. Load the dataset:**

In [ ]:
from pathlib import Path
import glob
import json
import pandas as pd


def load_sapsam_raw_model(
    csv_dir: Path,
    json_col = "Model JSON",
    csv_name = None,
    row_index = None,
):
    if not csv_dir.exists():
        raise FileNotFoundError(f"Dossier introuvable: {csv_dir}")

    #user selected model
    if csv_name is not None and row_index is not None:
        target_csv = csv_dir / csv_name
        if not target_csv.exists():
            raise FileNotFoundError(f"CSV introuvable: {target_csv}")

        df = pd.read_csv(target_csv)
        if json_col not in df.columns:
            raise KeyError(f"Colonne absente: '{json_col}' dans {csv_name}")

        if row_index < 0 or row_index >= len(df):
            raise IndexError(f"row_index hors limites: {row_index} (0..{len(df)-1})")

        raw = df.loc[row_index, json_col]
        if pd.isna(raw) or (isinstance(raw, str) and not raw.strip()):
            raise ValueError(f"JSON vide a {csv_name}:{row_index}")

        model = json.loads(raw) if isinstance(raw, str) else raw
        if not isinstance(model, dict):
            raise ValueError(f"Le JSON n'est pas un dict a {csv_name}:{row_index}")

        return model, csv_name, row_index

    # auto-pick first valid model
    csv_files = sorted(glob.glob(str(csv_dir / "*.csv")))
    if not csv_files:
        raise FileNotFoundError(f"Aucun CSV trouve dans: {csv_dir}")

    for csv_path in csv_files:
        df = pd.read_csv(csv_path)
        if json_col not in df.columns:
            continue

        for i in range(len(df)):
            raw = df.loc[i, json_col]
            if pd.isna(raw) or (isinstance(raw, str) and not raw.strip()):
                continue

            try:
                model = json.loads(raw) if isinstance(raw, str) else raw
                if isinstance(model, dict):
                    return model, Path(csv_path).name, i
            except Exception:
                continue

    raise RuntimeError("Aucun modele JSON valide trouve dans les CSV Sap-SAM.")


csv_dir = ROOT / "SAP_SAM_BPMN" / "sap_sam_2022" / "models"

# force a specific model
select = True
select_csv = "40000.csv"
select_raw = 0

if select:
    raw_model, picked_csv, picked_row = load_sapsam_raw_model(
        csv_dir=csv_dir,
        json_col="Model JSON",
        csv_name=select_csv,
        row_index=select_raw,
    )
else:
    raw_model, picked_csv, picked_row = load_sapsam_raw_model(
        csv_dir=csv_dir,
        json_col="Model JSON",
    )

print("CSV source:", picked_csv)
print("Row index:", picked_row)
print("Keywords:", list(raw_model.keys())[:10])
print("JSON Model:\n", raw_model)

**B. Extract BPMN structure (tasks, nodes, edges):**

In [ ]:
sap_dir = ROOT / "SAP_SAM_BPMN"
if str(sap_dir) not in sys.path:
    sys.path.insert(0, str(sap_dir))

bpmn_parser = load_module("bpmn_parser_module", sap_dir / "bpmn_parser.py")
bpmn = bpmn_parser.extract_bpmn(raw_model)



print("\n POOLS:")
for p in bpmn["pools"].values():
    print("-", p["name"])


print("\n\n LANES:")
for l in bpmn["lanes"].values():
    print("-", l["name"], "| pool:", bpmn["pools"][l["pool"]]["name"])

print("\n\nTASKS:")
for t in bpmn.get("tasks", {}).values():
    name = " ".join(t.get("name", "").split())
    doc = t.get("documentation", "")
    task_type = t.get("task_type", "")
    print(f"- {name} [{task_type}]")
    if doc:
        clean_doc = " ".join(str(doc).split())
        print("   doc:", clean_doc)

print("\n\nSEQUENCE EDGES:")
for e in bpmn.get("edges", {}).values():
    if e.get("type") != "SequenceFlow":
        continue
    src = e.get("src")
    tgt = e.get("tgt")
    if not src or not tgt:
        continue
    print(f"{bpmn_parser.pretty_name(bpmn, src)} -> {bpmn_parser.pretty_name(bpmn, tgt)}")


**C. Transform into SysML:**

In [ ]:
bpmn_into_sysml = load_module("bpmn_into_sysml_module", sap_dir / "bpmn_into_sysml.py")

sysml_code = bpmn_into_sysml.sysml_code(bpmn)

print("=" * 10)
print("GENERATED SYSML CODE")
print("=" * 10)
print(sysml_code)


### 4. Transformation of SOPBench :

**A. Load the dataset:**

In [ ]:
sop_main = load_module("sop_main_module", ROOT / "SOPBench" / "run_transformation.py")
sop_data = sop_main.load_sopbench_hf(split="train")

print("Nombre d'instances train:", len(sop_data))
print("Champs:", sop_data.column_names)

**B. Extract user prompt, source, nodes and edges:**

In [ ]:
row_index = 100
sample = sop_data[row_index]

graph = sop_main.extract_sopbench_raw(sample)

print("Description du graphe:")
print(graph["graph_description"])

print("\nNodes:")
for n in graph["nodes"]:
    print(n["id"], ":", n["name"], "| kind =", n["kind"], "| desc =", n["description"])

print("\nEdges:")
for src, dst in graph["edges"]:
    print(src, "->", dst)

**C. Transform into SysML:**

In [ ]:
transformed = sop_main.transform_or_nodes_to_fallback(graph)

pkg_name = sop_main.sanitize_name(str(sample.get("domain") or "SOPBenchPackage"))
process_name = sop_main.sanitize_name(str(sample.get("user_goal") or "Workflow"))

sysml_code = sop_main.build_sysml_from_transformed(
    transformed,
    package_name=pkg_name,
    action_def_name=process_name,
    agent_part_name="WorkflowAgent",
    agent_instance_name="workflowAgent",
)

print(sysml_code)